# Energy Balance time series evaluation at SNOTEL sites

In [ ]:
from pathlib import PurePath
import sys
import os
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import pyproj

import pandas as pd

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

import seaborn as sns
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

## Directory

In [ ]:
basin = 'yampa'
WY = 2021
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/data_extracts'

In [ ]:
snotel_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

# Basin polygon file
poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]

# SNOTEL all sites geojson fn
allsites_fn = h.fn_list(snotel_dir, 'snotel_sites_32613.json')[0]

# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=allsites_fn, buffer=200)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

ST_arr = ['CO'] * len(sitenums)
gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=int(WY), return_meta=True)

In [ ]:
short_sitenames = [''.join('_'.join(''.join(f.split(' ')).split('(')).split(')')) for f in sitenames]

# Read in pre-existing extracts

In [ ]:
# For a specific water year
extract_fns = h.fn_list(workdir, f'*{basin}*{WY}*.*')
_ = [print(f) for f in extract_fns]

In [ ]:
# Load the files anew (for all available water years, dependent on successful extraction)
# Note that net solar time is set at 00:00 UTC, so need to shift by 22 hours for mean daily values of em terms
dswrf_list = [xr.open_dataset(f).resample(time='1D').mean() for f in h.fn_list(workdir, f'DSWRF*{basin}*')]
em_list = [xr.open_dataset(f) for f in h.fn_list(workdir, f'*{basin}*em*')]
hm_netsolar_list= [xr.open_dataset(f).resample(time='1D').mean() for f in h.fn_list(workdir, f'net_HRRR_SPIReS*{basin}*_snotel.nc')]
baseline_netsolar_list = [xr.open_dataset(f).resample(time='1D').mean() for f in h.fn_list(workdir, f'*{basin}*smrf_energy_balance*')] # make sure this is extracted separately for plotting
print(len(dswrf_list), len(em_list), len(hm_netsolar_list), len(baseline_netsolar_list))

In [ ]:
# For the full time series, plot each year's variables on the same plot
# Loop through each variable
ylims = -20, 500
for data_var in ['DSWRF', 'net_rad', 'net_solar']:
# for data_var in ['net_rad', 'net_solar']:
    print(data_var)
    fig, axa = plt.subplots(1*len(sitenames), 1, figsize=(10,2*len(sitenames)), sharex=True, sharey=True)
    # Loop through sites
    for sdx, sitename in enumerate(sitenames):
        ax = axa[sdx]
        # Plot DSWRF
        if data_var == 'DSWRF':
            for ds in dswrf_list:
                wy = str(ds['time'][-1].values).split('-')[0]
                ds[data_var][:, sdx, sdx].plot(ax=ax, label=f'Raw DSWRF {wy}',
                                    #  color='lightgrey',
                                     linewidth=0.75, linestyle='-')
            for ds in hm_netsolar_list:
                wy = str(ds['time'][-1].values).split('-')[0]
                ds[data_var][:, sdx, sdx].plot(ax=ax, label=f'net HRRR SPIReS {wy}',
                                     color='red', marker='.', markersize=2,
                                     linestyle='None')
            ax = h.prep_axis(ax=ax, ylims=ylims, fc=None, xlab='', ylab='Wm-2')
        # Plot net radiation
        elif data_var == 'net_rad':
            halfway = len(em_list) // 2
            for ds_baseline, ds_hm in zip(em_list[:halfway], em_list[halfway:]):
                wy = str(ds_baseline['time'][-1].values).split('-')[0]
                ds_baseline[data_var][:, sdx, sdx].plot(ax=ax, label=f'Baseline {wy}',
                                                        #  color='red',
                                                         linewidth=1, linestyle='-')
                ds_hm[data_var][:, sdx, sdx].plot(ax=ax, label=f'HRRR-SPIReS {wy}', color='red', linewidth=1, linestyle='-')
            ax = h.prep_axis(ax=ax, ylims=(-100, 400), fc=None, xlab='', ylab='Wm-2')
        # Plot albedo and net solar radiation
        elif data_var == 'net_solar':
            for ds_baseline, ds_hm in zip(baseline_netsolar_list, hm_netsolar_list):
                wy = str(ds_baseline['time'][-1].values).split('-')[0]
                ds_baseline[data_var][:, sdx, sdx].plot(ax=ax, label=f'Baseline {wy}',
                                                        # color='red',
                                                        linewidth=1, linestyle='-')
                ds_hm[data_var][:, sdx, sdx].plot(ax=ax, label=f'HRRR-SPIReS {wy}', color='red',linewidth=1, linestyle='-')
            ax = h.prep_axis(ax=ax, ylims=(-100, 400), fc=None, xlab='', ylab='Wm-2')
        if sdx == 0:
            ax.set_title(f'{basin.capitalize()} {data_var}')
        else:
            ax.set_title('')
        ax.set_xlabel('')
        ax.annotate(sitename, xy=(0.01, 0.84), xycoords='axes fraction', ha='left', fontsize=10, fontstyle='italic')
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.2), fontsize=10, frameon=False)
        ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
plt.tight_layout()
# Plot snow depth (hm, baseline, snotel) on the second axis for net solar and net radiation plots (can also do this in another plot below)

# Longwave

In [ ]:
# Where are spikes in net radiation coming from? Check longwave
# Plot longwave for HM and Baseline (smrf - thermal)
# Load the files anew
baseline_lw_list = [xr.open_dataset(f).resample(time='1D').mean() for f in h.fn_list(workdir, f'*{basin}_Baseline_smrf_2*')]
hm_lw_list = [xr.open_dataset(f).resample(time='1D').mean() for f in h.fn_list(workdir, f'*{basin}_HRRR-SPIReS_smrf_2*')]
print(len(baseline_lw_list), len(hm_lw_list))

In [ ]:
data_var = 'thermal'
# Loop through sites
fig, axa = plt.subplots(len(sitenames), 1, figsize=(10,2*len(sitenames)), sharex=True, sharey=True)
for sdx, sitename in enumerate(sitenames):
    ax = axa[sdx]
    # fig, ax = plt.subplots(1, 1, figsize=(10,2), sharex=True, sharey=True)
    for ds in baseline_lw_list:
        wy = str(ds['time'][-1].values).split('-')[0]
        ds[data_var][:, sdx, sdx].plot(ax=ax, label=f'{wy}',
                            #  color='lightgrey',
                                linewidth=0.75, linestyle='-')
    for ds in hm_lw_list:
        ds[data_var][:, sdx, sdx].plot(ax=ax, label=f'{wy}',
                                color='red', marker='.', markersize=2,
                                linestyle='None')
    ylims = (100, 400)
    ax = h.prep_axis(ax=ax, ylims=ylims, fc=None, xlab='', ylab='Wm-2')
    ax.set_title('')
    ax.set_xlabel('')
    ax.annotate(sitename, xy=(0.01, 0.84), xycoords='axes fraction', ha='left', fontsize=11, fontstyle='italic')
    ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    if sdx == 0:
        ax.set_title(f'{basin.capitalize()} {data_var}')
    else:
        ax.set_title('')
plt.tight_layout()

In [ ]:
# get cmap from palette
cmap = sns.color_palette(n_colors=7, palette='plasma')
cmap

In [ ]:
figdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures'

In [ ]:
year_jdx = 0
from matplotlib import patheffects
data_vars = ['sum_EB', 'net_rad', 'net_solar', 'sensible_heat', 'latent_heat', 'snow_soil', 'precip_advected']
num_plots = len(data_vars)
for sdx, sitename in enumerate(sitenames):
    fig, axa = plt.subplots(num_plots, 1, figsize=(8, 1.2*num_plots), sharex=True, sharey=True)
    for jdx, f in enumerate(data_vars):
        ax = axa[jdx]
        if jdx == 2:
            baseline_netsolar_list[0 + year_jdx][f][:, sdx, sdx].plot(ax=ax, label='Baseline', color=cmap[jdx], linewidth=3, alpha=0.4)
            hm_netsolar_list[0 + year_jdx][f][:, sdx, sdx].plot(ax=ax, label='HRRR-SPIReS', color=cmap[jdx], linewidth=1)
        else:
            em_list[0 + year_jdx][f][:, sdx, sdx].plot(ax=ax, label='Baseline', color=cmap[jdx], linewidth=3, alpha=0.4)
            em_list[4 + year_jdx][f][:, sdx, sdx].plot(ax=ax, label='HRRR-SPIReS', color=cmap[jdx], linewidth=1)
        # Annotate f in upper lefthand corner inside plot and add white buffer
        ax.annotate(f, xy=(0.985, 0.8), xycoords='axes fraction', ha='right', c=cmap[jdx], fontsize=10,
                    path_effects=[patheffects.withStroke(linewidth=5, foreground="w")])
        ax.set_xlabel('')
        ax.set_ylabel('W m-2')
        # Trim everything to August
        xmin, xmax = pd.Timestamp(f'{WY-1 + year_jdx}-10-01'), pd.Timestamp(f'{WY + year_jdx}-7-15')
        # Format limits and title
        ax.set_xlim(xmin, xmax)
        ax.set_title('')
        ax.set_ylim(-100, 200)
        # Add zero line
        ax.axhline(0, color='black', linewidth=0.5)
        # Add grid
        ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    plt.suptitle(f'{basin.capitalize()} WY {WY + year_jdx} - {sitename}', fontsize=11, fontstyle='italic')
    plt.tight_layout()
    figname = f'{figdir}/{basin}_wy{WY + year_jdx}_{short_sitenames[sdx]}_energy_balance_terms_daily.png'
    print(figname)
    plt.savefig(figname, dpi=300)

In [ ]:
def resample_to_monthly_and_trim(ds):
    ds = ds.resample(time='ME').mean()
    # Get the year first
    thisyear = ds['time'].dt.year.min().values

    # Then set the date range
    start_date = f'{thisyear}-10-01'
    end_date = f'{thisyear+1}-07-31'
    print(ds.time[0].values, ds.time[-1].values)

    # Trim the dataset to the specified date range
    ds = ds.sel(time=slice(start_date, end_date))
    print(ds.time[0].values, ds.time[-1].values, '\n')
    return(ds)

### Resample to monthly and trim to October 1 to July 31

In [ ]:
em_list_monthly_resampled = []
for ds in em_list:
    ds = resample_to_monthly_and_trim(ds)
    # # Fix the time
    # ds['time'] = ds['time'].dt.date
    em_list_monthly_resampled.append(ds)

In [ ]:
net_solar_monthly_resampled = []
for ds_baseline in baseline_netsolar_list:
    ds_baseline = resample_to_monthly_and_trim(ds_baseline)
    net_solar_monthly_resampled.append(ds_baseline)
for ds_hs in hm_netsolar_list:
    ds_hs = resample_to_monthly_and_trim(ds_hs)
    net_solar_monthly_resampled.append(ds_hs)

# Add fig saving for daily and monthly eb plots, modify existing energy balance plotting script

In [ ]:
from matplotlib import patheffects
data_vars = ['sum_EB', 'net_rad', 'net_solar', 'sensible_heat', 'latent_heat', 'snow_soil', 'precip_advected']
num_plots = len(data_vars)
for sdx, sitename in enumerate(sitenames):
    fig, axa = plt.subplots(num_plots, 1, figsize=(6, 1.2*num_plots), sharex=True, sharey=True)
    for jdx, f in enumerate(data_vars):
        ax = axa[jdx]
        if jdx == 2:
            net_solar_monthly_resampled[0 + year_jdx][f][:, sdx, sdx].to_series().plot.bar(label='Baseline', ax=ax, color=cmap[jdx], alpha=0.5)
            net_solar_monthly_resampled[4 + year_jdx][f][:, sdx, sdx].to_series().plot.bar(label='HRRR-SPIReS', color=cmap[jdx], alpha=0.5, ax=ax)
            ds = net_solar_monthly_resampled[0 + year_jdx]
        else:
            # Plot the monthly means for each variable
            em_list_monthly_resampled[0 + year_jdx][f][:, sdx, sdx].to_series().plot.bar(label='Baseline', ax=ax, color=cmap[jdx], alpha=1)
            em_list_monthly_resampled[4 + year_jdx][f][:, sdx, sdx].to_series().plot.bar(label='HRRR-SPIReS', color=cmap[jdx], alpha=0.5, ax=ax)
            ds = em_list_monthly_resampled[0 + year_jdx]
        # Annotate f in upper lefthand corner inside plot and add white buffer
        ax.annotate(f, xy=(0.985, 0.8), xycoords='axes fraction', ha='right', c=cmap[jdx], fontsize=10,
                    path_effects=[patheffects.withStroke(linewidth=5, foreground="w")])
        ax.set_xlabel('')
        ax.set_ylabel('W m-2')
        # Rename the x-axis tick labels to year and month
        ax.set_xticklabels([ds['time'].dt.strftime('%Y-%m').values[i] for i in range(len(ds['time']))], rotation=45)
        ax.set_title('')
        ax.set_ylim(-50, 150)
        # Add zero line
        ax.axhline(0, color='black', linewidth=0.5)
        # Add grid
        ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    plt.suptitle(sitename, fontsize=11, fontstyle='italic')
    plt.tight_layout()

In [ ]:
def build_dict(em_list_monthly_resampled, net_solar_monthly_resampled, year_jdx, sitenames,
               data_vars=['sum_EB', 'net_rad', 'net_solar', 'sensible_heat', 'latent_heat', 'snow_soil', 'precip_advected']):
    """TODO: need to fix input resampled data for the number of processed water years, probably should just pull anew for that specific WY"""
    # convert to file compatible names
    short_sitenames = [''.join('_'.join(''.join(f.split(' ')).split('(')).split(')')) for f in sitenames]
    # Put all energy balance terms into a large dict
    data_dict = dict()
    for jdx, f in enumerate(data_vars):
        data_var = data_vars[jdx]
        # Get the monthly means for each site
        for sdx, short_sitename in enumerate(short_sitenames):
            if jdx == 2:
                data_dict[f'{short_sitename}_baseline_{data_var}'] = net_solar_monthly_resampled[0 + year_jdx][f][:, sdx, sdx].to_series()
                data_dict[f'{short_sitename}_hs_{data_var}'] = net_solar_monthly_resampled[4 + year_jdx][f][:, sdx, sdx].to_series()
            else:
                data_dict[f'{short_sitename}_baseline_{data_var}'] = em_list_monthly_resampled[0 + year_jdx][f][:, sdx, sdx].to_series()
                data_dict[f'{short_sitename}_hs_{data_var}'] = em_list_monthly_resampled[4 + year_jdx][f][:, sdx, sdx].to_series()
    return data_dict, short_sitenames

In [ ]:
def select_data(data_dict, data_var):
    selected_data = [data_dict[f] for f in data_dict.keys() if data_var in f]
    selected_keys = [f for f in data_dict.keys() if data_var in f]
    selected_data_df = pd.DataFrame(selected_data, index=selected_keys).T
    return selected_data_df

In [ ]:
sns.set_palette('tab20')

In [ ]:
def plot_grouped_ebterms(baseline, hs, basin, WY, data_var, figsize=(10, 5)):
    # Plot the data in grouped bars based on month for the selected keys
    # Baseline
    _, ax = plt.subplots(1, 1, figsize=figsize, sharex=True, sharey=True)
    baseline.plot.bar(ax=ax, alpha=1)

    # Rename the x-axis tick labels to year and month
    ax.set_xticklabels(baseline.index.strftime('%Y-%m'), rotation=45)
    ax.set_ylim(-50, 150)
    # Add zero line
    ax.axhline(0, color='black', linewidth=0.5)
    # Add grid
    ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    ax.set_xlabel('')
    ax.set_ylabel('W m-2')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, frameon=False)
    ax.set_title(f'Monthly mean {basin.capitalize()} {WY} {data_var}')

    # HRRR-SPIReS
    _, ax = plt.subplots(1, 1, figsize=figsize, sharex=True, sharey=True)
    hs.plot.bar(ax=ax, alpha=1)

    # Rename the x-axis tick labels to year and month
    ax.set_xticklabels(hs.index.strftime('%Y-%m'), rotation=45)
    ax.set_ylim(-50, 150)
    # Add zero line
    ax.axhline(0, color='black', linewidth=0.5)
    # Add grid
    ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    ax.set_xlabel('')
    ax.set_ylabel('W m-2')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, frameon=False)


In [ ]:
year_jdx = 0
data_dict, short_sitenames = build_dict(em_list_monthly_resampled, net_solar_monthly_resampled, year_jdx, sitenames)
data_var = 'sum_EB'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_ebterms(baseline, hs, basin, WY + year_jdx, data_var)

In [ ]:
data_var = 'net_rad'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_ebterms(baseline, hs, basin, WY + year_jdx, data_var)

In [ ]:
data_var = 'net_solar'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_ebterms(baseline, hs, basin, WY + year_jdx, data_var)

In [ ]:
def plot_grouped_eb_singleplot(baseline, hs, basin, WY, data_var, figsize=(10, 5), legend_on=False):
    # Plot the data in grouped bars based on month for the selected keys on a single plot
    # Baseline
    _, ax = plt.subplots(1, 1, figsize=figsize)
    baseline.plot.bar(ax=ax, alpha=1)

    # HRRR-SPIReS
    # Turn off bar facecolor
    hs.plot.bar(ax=ax, edgecolor='black', facecolor='none', linewidth=0.5)
    # Rename the x-axis tick labels to year and month
    ax.set_xticklabels(hs.index.strftime('%Y-%m'), rotation=45)
    ax.set_ylim(-50, 150)
    # Add zero line
    ax.axhline(0, color='black', linewidth=0.5)

    # Add grid, fix labels, set title
    ax.grid(color='grey', linestyle='--', linewidth=1, which='both', alpha=0.3)
    ax.set_xlabel('')
    ax.set_ylabel('W m-2')
    ax.set_title(f'Monthly mean {basin.capitalize()} {WY} {data_var}')

    # Only do the first half of the legend (not the second plot)
    if legend_on:
        # Get the handles and labels of the legend
        handles, labels = ax.get_legend_handles_labels()

        # Calculate the midpoint for selecting half the items
        half_point = len(handles) // 2

        # Create a new legend with the first half of the items
        ax.legend(handles[:half_point], labels[:half_point], loc='upper left', bbox_to_anchor=(0, 1), fontsize=10, frameon=True)
    else:
        # Turn off the legend
        ax.legend().remove()

In [ ]:
year_jdx = 3
data_dict, short_sitenames = build_dict(em_list_monthly_resampled, net_solar_monthly_resampled, year_jdx, sitenames)
data_var = 'sum_EB'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_eb_singleplot(baseline, hs, basin, WY + year_jdx, data_var, legend_on=True)

In [ ]:
data_var = 'net_rad'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_eb_singleplot(baseline, hs, basin, WY + year_jdx, data_var)

In [ ]:
data_var = 'net_solar'
baseline = select_data(data_dict, f'baseline_{data_var}')
hs = select_data(data_dict, f'hs_{data_var}')
plot_grouped_eb_singleplot(baseline, hs, basin, WY + year_jdx, data_var)

In [ ]:
# Plot up just longwave, net solar, and net radiation
